# Data collection, parsing and preprocessing
**David · Cambridge / Bank of England employer project**

I assembled the R1C5 earnings-call corpus for BAC, Citi and JPM: 60 calls across 2021Q1–2025Q4, plus one historical BAC source revision.

The pipeline is: source registration → text extraction → structural parsing → canonical tables → research views. This notebook documents that pipeline, checks the frozen inputs and exposes the rebuild function. The executed outputs report source-integrity and corpus-inspection checks. The exact historical rebuild is pending the Git object recorded in Section 7.


## 1. Setup and release identity

I use the private source package and production modules under `src/cam_boe/` and `scripts/`. The hashes below bind these checks to the accepted R1C5 release. Execution uses those private files and the project environment. The tables are saved in the notebook for review.


In [1]:
from pathlib import Path
import hashlib
import json
import subprocess
import tempfile

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display, HTML

def show_corpus_table(frame, caption):
    # Show every row; wide schema tables can scroll horizontally.
    display(HTML('<div style="overflow-x:auto;max-width:100%">'
                 + frame.to_html(border=0, escape=True, na_rep="Not applicable") + '</div>'))
    print(caption)

# Locate the private project code and source package.
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "cam_boe").exists() and (candidate / "team_data").exists():
            return candidate
        nested = candidate / "david" / "cambridge_boe_r1c5"
        if (nested / "src" / "cam_boe").exists() and (nested / "team_data").exists():
            return nested
    raise FileNotFoundError("Could not locate david/cambridge_boe_r1c5")

ROOT = find_project_root()
PACKAGE = ROOT / "team_data/CAM_BOE_TEAM_DATA_RELEASE_R1C5_CANDIDATE_V1"
RELEASE_ID = "personal-r1c5-v1-f43132bb41508cb3"
RELEASE = PACKAGE / "processed" / RELEASE_ID

# Pin the accepted package, parser and build identity.
EXPECTED = {
    "team_manifest_sha256": "c1ee35c3e8bb23288d6478627a357a2d84021726635b2d037ec690ac45a05e6a",
    "release_manifest_sha256": "ec7313cba9a320b3e61d3ca926a507c5eaf9ff67bb00e137d662805e0b9de89f",
    "release_payload_sha256": "5e360c511a9240f4238e329a9d83fd54ae837e385ea92938d084efd8f3259d39",
    "release_identity_sha256": "f43132bb41508cb3aaa2e248937d5c49b54b95e3d55558209d2fa831c32db0ba",
    "parser_fingerprint": "22ce785c744ebb2fe674cab7e9c6edf1741a0887761da689a2b7a1778454ca0b",
    "immutable_build_commit": "279b180cf6096857c9e379554785adb3470c11e4",
}

# Hash large files in fixed-size chunks.
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024*1024), b""):
            h.update(block)
    return h.hexdigest()

# Verify manifest identity; source files are checked below.
release_manifest = json.loads((RELEASE / "release_manifest.json").read_text())

assert sha256(PACKAGE / "TEAM_DATA_RELEASE_MANIFEST.json") == EXPECTED["team_manifest_sha256"]
assert sha256(RELEASE / "release_manifest.json") == EXPECTED["release_manifest_sha256"]
assert release_manifest["release_payload_sha256"] == EXPECTED["release_payload_sha256"]
assert release_manifest["identity_material_sha256"] == EXPECTED["release_identity_sha256"]
assert release_manifest["candidate_release_authority"]["parser_fingerprint"] == EXPECTED["parser_fingerprint"]

print("Accepted release:", RELEASE_ID)
print("Accepted release identity:", EXPECTED["release_identity_sha256"])


Accepted release: personal-r1c5-v1-f43132bb41508cb3
Accepted release identity: f43132bb41508cb3aaa2e248937d5c49b54b95e3d55558209d2fa831c32db0ba


## 2. Collection and source registration

I retain 61 registered revisions: 20 current BAC HTML sources, 20 Citi PDFs, 20 JPM PDFs and one historical BAC 2025Q2 revision. Each record links the document to its hashes, extraction and acquisition metadata.

The original collection used the project's source-audit, approved-download and local-import interfaces. The catalogue preserves those collection events. Reproduction uses the frozen files offline.


In [2]:
source_revisions = pd.read_parquet(
    RELEASE / "A_catalogue_and_provenance/source_revisions.parquet"
)

# Keep the historical revision separate from current calls.
source_summary = (
    source_revisions
    .groupby(["bank_id", "media_type", "current_source_revision"], dropna=False)
    .size()
    .rename("source_revisions")
    .reset_index()
)

show_corpus_table(source_summary, "Table C1. Registered sources by bank, format and current/history status.")

print("All revisions:", len(source_revisions))
print("Current revisions:", int(source_revisions["current_source_revision"].eq(True).sum()))
print("History-only revisions:", int(source_revisions["current_source_revision"].eq(False).sum()))
print("Current calls:", source_revisions.loc[source_revisions["current_source_revision"].eq(True), "call_id"].nunique())


,bank_id,media_type,current_source_revision,source_revisions
0,BAC,text/html,False,1
1,BAC,text/html,True,20
2,CITI,application/pdf,True,20
3,JPM,application/pdf,True,20


Table C1. Registered sources by bank, format and current/history status.
All revisions: 61
Current revisions: 60
History-only revisions: 1
Current calls: 60


## 3. Source integrity

I verify each revision's raw file, extracted text and extraction diagnostics. Raw/extracted hashes must match the catalogue, and the diagnostic record must bind the same source and extraction IDs. Table C2 reports 61 checked revisions and zero integrity problems.


In [3]:
def verify_source_triplets(rows, package_root):
    problems=[]
    checked=0
    for row in rows.to_dict("records"):
        # Resolve the three source-bound artifacts.
        raw = package_root / row["local_relative_path"]
        extracted = package_root / row["extracted_relative_path"]
        diagnostics = package_root / row["extraction_diagnostics_relative_path"]

        # Report missing files before checking their contents.
        for label, path in [("raw", raw), ("extracted", extracted), ("diagnostics", diagnostics)]:
            if not path.is_file():
                problems.append((row["source_revision_id"], f"missing_{label}"))
                continue

        # Match the original raw and extracted hashes.
        if raw.is_file() and sha256(raw) != row["sha256"]:
            problems.append((row["source_revision_id"], "raw_hash_mismatch"))
        if extracted.is_file() and sha256(extracted) != row["extracted_sha256"]:
            problems.append((row["source_revision_id"], "extracted_hash_mismatch"))

        # Cross-check diagnostic IDs against the catalogue.
        if diagnostics.is_file():
            diag=json.loads(diagnostics.read_text())
            expected_bindings={
                "source_record_id": row["source_record_id"],
                "source_sha256": row["sha256"],
                "extraction_id": row["extraction_id"],
                "text_sha256": row["extracted_sha256"],
            }
            observed={k:diag.get(k) for k in expected_bindings}
            if observed != expected_bindings:
                problems.append((row["source_revision_id"], "diagnostic_binding_mismatch"))
        checked += 1

    return {"checked_revisions": checked, "problems": problems}

# Validate all 61 registered revisions.
source_integrity = verify_source_triplets(source_revisions, PACKAGE)
show_corpus_table(pd.Series({
    "source revisions checked": source_integrity["checked_revisions"],
    "integrity problems": len(source_integrity["problems"]),
}).to_frame("value"), "Table C2. Raw/extracted hashes and diagnostic bindings.")
assert source_integrity["problems"] == []


,value
source revisions checked,61
integrity problems,0


Table C2. Raw/extracted hashes and diagnostic bindings.


## 4. Text extraction

I use `scripts/extract_text.py` for the generic extraction route. Registered-source reprocessing is handled by `reprocess_registered_personal_intake()`, which performs offline extraction and parsing.

Source and extracted-text hashes, extraction IDs and source coordinates remain attached to the outputs. These preserve lineage back to the document.


## 5. Structural parsing

I use a deterministic, rule-based parser and retain uncertain classifications.

| Layer | Values |
|---|---|
| Section | prepared_remarks, qa, other, unknown |
| Role | management, analyst, operator, other, unknown |
| Channel | prepared_management, analyst_question, management_answer, other, unknown |

Management roles use call-specific participant titles; analyst roles use operator introductions and matching speaker labels; explicit labels identify operators. Ambiguous cases remain in the review queue.

Question units are analyst-turn proxies, so a multipart question can remain one unit. Sentence splitting is offline and handles financial abbreviations, initials, decimals and percentages.


In [4]:
# Read the accepted parser configuration.
authority = release_manifest["candidate_release_authority"]
parser_prov = authority["parser_provenance"]

parser_view = pd.Series({
    "parser version": authority["parser_version"],
    "parser fingerprint": authority["parser_fingerprint"],
    "canonical schema": parser_prov["canonical_schema"]["version"],
    "canonical schema fingerprint": parser_prov["canonical_schema"]["fingerprint"],
    "layout router": parser_prov["layout_router"]["version"],
    "layout registry": parser_prov["layout_registry"]["schema_version"],
    "structural adapter": parser_prov["structural_adapter"]["version"],
    "coordinate contract": parser_prov["coordinates"],
    "external model used": False,
})
show_corpus_table(parser_view.to_frame("value"), "Table C3. Parser and coordinate configuration.")


,value
parser version,cam_boe.transcript_parser.layout_family_adapter.v1
parser fingerprint,22ce785c744ebb2fe674cab7e9c6edf1741a0887761da689a2b7a1778454ca0b
canonical schema,cam_boe.canonical_transcript_tables.v1
canonical schema fingerprint,8428afe6d9341fd720a38600673a9113c7aa343c865fb7b99ba932c3258be623
layout router,cam_boe.layout_family_router.v1
layout registry,cam_boe.layout_family_registry.v1
structural adapter,cam_boe.structural_overlay_adapter.v1
coordinate contract,cam_boe.extraction_codepoints.half_open.segments.v2
external model used,False


Table C3. Parser and coordinate configuration.


## 6. Pipeline implementation

The following production modules implement the stages called by the rebuild function.

| Stage | File |
|---|---|
| Registration / offline reprocessing | `src/cam_boe/personal_acquisition.py` |
| Transcript parsing | `src/cam_boe/parsing/transcripts.py` |
| Layout routing | `src/cam_boe/parsing/layout_families.py` |
| Structural overlays | `src/cam_boe/parsing/structural_overlays.py` |
| Source coordinates | `src/cam_boe/parsing/source_mapping.py` |
| Schemas | `src/cam_boe/schemas.py` |
| Corpus assembly | `src/cam_boe/personal_corpus.py` |
| Controlled reparse | `src/cam_boe/controlled_reparse.py` |
| Candidate build | `scripts/r1c5_candidate_release.py` |
| Package validation / loading | `src/cam_boe/team_data_release.py`, `scripts/team_data_release.py` |


## 7. Exact-rebuild recipe

I reconstruct the registered source state, copy its raw/extracted/diagnostic files into a temporary workspace, reparse offline, check that the source files are unchanged, and assemble the release twice. The rebuild must match the accepted release hashes and reuse an identical result on the second assembly.

**Open dependency:** the historical build commit `279b180cf6096857c9e379554785adb3470c11e4` is absent from this clone. I leave `RUN_EXACT_REBUILD=False`. The historical repository and private dependencies must be restored before validating the complete path. The optional demonstration uses a temporary workspace and discards rebuilt files when it exits.


In [5]:
# Inspection mode; leave the unavailable historical rebuild off.
RUN_EXACT_REBUILD = False

def git_has_commit(project_root, commit):
    result = subprocess.run(
        ["git", "cat-file", "-e", f"{commit}^{{commit}}"],
        cwd=project_root,
        capture_output=True,
    )
    return result.returncode == 0

# Check the historical Git object before enabling any rebuild.
exact_rebuild_prerequisite = git_has_commit(ROOT, EXPECTED["immutable_build_commit"])

print("Immutable build commit available:", exact_rebuild_prerequisite)
print("Exact rebuild enabled:", RUN_EXACT_REBUILD)

if RUN_EXACT_REBUILD and not exact_rebuild_prerequisite:
    raise RuntimeError(
        "Exact R1C5 rebuild requires the historical immutable build commit. "
        "Restore the private repository history, then rerun this cell."
    )


Immutable build commit available: False
Exact rebuild enabled: False


In [6]:
def rebuild_exact_r1c5(workspace_root):
    from cam_boe.controlled_reparse import (
        reconstruct_t2_records,
        _copy_registered_assets,
        source_asset_snapshot,
    )
    from cam_boe.workspace import (
        build_workspace_paths,
        initialize_workspace,
        load_workspace_paths,
    )
    from cam_boe.personal_acquisition import (
        reprocess_registered_personal_intake,
        load_personal_intake_manifest,
    )
    from cam_boe.personal_corpus import (
        assemble_personal_corpus,
        candidate_release_authority,
    )

    workspace_root = Path(workspace_root).resolve()

    # Recover registered inputs before the later selection overlay.
    source_rows = pq.read_table(
        RELEASE / "A_catalogue_and_provenance/source_revisions.parquet"
    ).to_pylist()
    records = reconstruct_t2_records(source_rows)

    # Create an isolated workspace; leave the accepted release untouched.
    paths = build_workspace_paths(workspace_root)
    initialize_workspace(paths, project_root=ROOT)
    loaded = load_workspace_paths(workspace_root, project_root=ROOT)
    data_root = loaded.output_root / "personal_intake"

    # Copy frozen sources and record their initial hashes.
    _copy_registered_assets(PACKAGE, data_root, records)
    before = source_asset_snapshot(data_root, records)

    # Re-extract/reparse the registered inventory offline.
    reparse = reprocess_registered_personal_intake(
        workspace_root,
        records,
        project_root=ROOT,
    )

    # Confirm raw and extracted source authority is unchanged.
    refreshed = load_personal_intake_manifest(
        workspace_root,
        project_root=ROOT,
    )
    after = source_asset_snapshot(data_root, refreshed)
    if before != after:
        raise RuntimeError("Source authority changed during offline reprocessing")

    # Bind the exact historical build commit.
    release_authority = candidate_release_authority(
        ROOT,
        build_source_commit=EXPECTED["immutable_build_commit"],
    )

    # Assemble canonical tables and research views.
    built = assemble_personal_corpus(
        workspace_root,
        project_root=ROOT,
        release_version="r1c5-v1",
        release_authority=release_authority,
    )

    # Repeat assembly to check deterministic reuse.
    repeat = assemble_personal_corpus(
        workspace_root,
        project_root=ROOT,
        release_version="r1c5-v1",
        release_authority=release_authority,
    )

    rebuilt_release = Path(built["release_directory"])
    rebuilt_manifest = json.loads(
        (rebuilt_release / "release_manifest.json").read_text()
    )

    # Exact identity requires all expected hashes to match.
    checks = {
        "release_id": built["release_id"] == RELEASE_ID,
        "release_manifest_sha256": built["release_manifest_sha256"]
            == EXPECTED["release_manifest_sha256"],
        "release_payload_sha256": built["release_payload_sha256"]
            == EXPECTED["release_payload_sha256"],
        "release_identity_sha256": rebuilt_manifest["identity_material_sha256"]
            == EXPECTED["release_identity_sha256"],
        "repeat_assembly": repeat["status"] == "REUSED_IDENTICAL_RELEASE",
        "source_preservation": before == after,
    }

    if not all(checks.values()):
        raise RuntimeError(f"Exact rebuild mismatch: {checks}")

    return {
        "reparse": reparse,
        "built": built,
        "repeat": repeat,
        "checks": checks,
    }

# This demonstration discards its temporary rebuilt files on exit.
if RUN_EXACT_REBUILD:
    with tempfile.TemporaryDirectory(prefix="r1c5_notebook_rebuild_") as td:
        rebuild_result = rebuild_exact_r1c5(Path(td) / "workspace")
        display(pd.Series(rebuild_result["checks"]).to_frame("PASS"))
else:
    print("Exact rebuild function defined but not executed in this checkout.")


Exact rebuild function defined but not executed in this checkout.


## 8. Corpus layout

I organise the release into six table groups:

| Group | Contents |
|---|---|
| A | Source catalogue, selections and coverage |
| B | Calls, documents, speakers, turns, exchanges, questions and sentences |
| C | Companion-evidence inventory |
| D | Analysis-ready research views |
| E | Quality observations and review queue |
| F | Retrieval records |

JSONL and Parquet are canonical; CSV is for inspection. Some larger exports use gzip. Table C4 lists all 34 logical tables and their row counts. Research views reuse canonical rows, so the total includes overlapping representations.


In [7]:
# Flatten the release catalogue; keep all 34 tables visible.
dataset_rows=[]
for logical_bank, bank_info in release_manifest["logical_banks"].items():
    for name, info in bank_info["datasets"].items():
        dataset_rows.append({
            "logical_bank": logical_bank,
            "table": name,
            "rows": info["row_count"],
            "parquet": info["files"]["parquet"],
        })

catalogue = pd.DataFrame(dataset_rows).sort_values(["logical_bank","table"])
show_corpus_table(catalogue, "Table C4. All logical tables; linked-view rows overlap.")

print("Logical tables:", catalogue["table"].nunique())
print("Total logical rows:", int(catalogue["rows"].sum()))


,logical_bank,table,rows,parquet
0,canonical_transcripts,call_documents,60,B_canonical_transcripts/call_documents.parquet
1,canonical_transcripts,calls,60,B_canonical_transcripts/calls.parquet
2,canonical_transcripts,documents,60,B_canonical_transcripts/documents.parquet
3,canonical_transcripts,exchange_turns,5039,B_canonical_transcripts/exchange_turns.parquet
4,canonical_transcripts,qa_exchanges,1900,B_canonical_transcripts/qa_exchanges.parquet
5,canonical_transcripts,question_units,1900,B_canonical_transcripts/question_units.parquet
6,canonical_transcripts,sentences,39738,B_canonical_transcripts/sentences.parquet
7,canonical_transcripts,speakers,873,B_canonical_transcripts/speakers.parquet
8,canonical_transcripts,turns,5342,B_canonical_transcripts/turns.parquet
9,catalogue_and_provenance,active_selections,60,A_catalogue_and_provenance/active_selections.parquet


Table C4. All logical tables; linked-view rows overlap.
Logical tables: 34
Total logical rows: 77070


## 9. Canonical checks

I check the 60-call bank–quarter grid and selected ID relationships using the canonical Parquet tables. The summary and complete grid are shown in Tables C5–C6.


In [8]:
# Load the canonical tables for count and key checks.
B = RELEASE / "B_canonical_transcripts"

calls = pd.read_parquet(B / "calls.parquet")
turns = pd.read_parquet(B / "turns.parquet")
speakers = pd.read_parquet(B / "speakers.parquet")
qa = pd.read_parquet(B / "qa_exchanges.parquet")
questions = pd.read_parquet(B / "question_units.parquet")
sentences = pd.read_parquet(B / "sentences.parquet")

# Reconstruct the complete bank–quarter grid.
coverage = (
    calls.groupby(["fiscal_year","fiscal_quarter","bank_id"])["call_id"]
    .nunique()
    .unstack("bank_id", fill_value=0)
)

checks = pd.Series({
    "calls": len(calls),
    "unique call IDs": calls["call_id"].nunique(),
    "banks": calls["bank_id"].nunique(),
    "quarter rows": coverage.shape[0],
    "complete bank-quarter cells": int((coverage > 0).sum().sum()),
    "turns": len(turns),
    "speakers": len(speakers),
    "Q&A exchanges": len(qa),
    "question units": len(questions),
    "sentences": len(sentences),
})
show_corpus_table(checks.to_frame("value"), "Table C5. Canonical corpus counts.")
show_corpus_table(coverage, "Table C6. Calls per bank and quarter.")

# Validate counts and selected foreign-key links.
assert len(calls) == 60
assert calls["call_id"].nunique() == 60
assert calls["bank_id"].nunique() == 3
assert coverage.shape[0] == 20
assert int((coverage > 0).sum().sum()) == 60
assert set(turns["call_id"]).issubset(set(calls["call_id"]))
assert set(questions["turn_id"]).issubset(set(turns["turn_id"]))
assert set(qa["call_id"]).issubset(set(calls["call_id"]))


,value
calls,60
unique call IDs,60
banks,3
quarter rows,20
complete bank-quarter cells,60
turns,5342
speakers,873
Q&A exchanges,1900
question units,1900
sentences,39738


Table C5. Canonical corpus counts.


Table C6. Calls per bank and quarter.


## 10. Research views

I provide prepared-management, analyst-question and management-Q&A views, with exchange context, common-quarter coverage and role/source diagnostics. These retain the canonical IDs so downstream models use the same populations. Table C7 summarises the views.


In [9]:
D = RELEASE / "D_research_views"

view_names = [
    "prepared_management_turns",
    "analyst_question_turn_proxies",
    "management_qa_turns",
    "ordered_exchange_context",
    "common_quarter_panel",
    "role_evidence_distribution",
    "role_evidence_sensitivity",
    "strict_provisional_sensitivity",
]

# One inventory row per reusable view.
view_summary=[]
for name in view_names:
    df=pd.read_parquet(D / f"{name}.parquet")
    view_summary.append({
        "view": name,
        "rows": len(df),
        "calls": df["call_id"].nunique() if "call_id" in df.columns else None,
    })

show_corpus_table(pd.DataFrame(view_summary), "Table C7. Analysis-ready views and call coverage.")


,view,rows,calls
0,prepared_management_turns,157,60.0
1,analyst_question_turn_proxies,1900,60.0
2,management_qa_turns,2403,60.0
3,ordered_exchange_context,5039,60.0
4,common_quarter_panel,20,Not applicable
5,role_evidence_distribution,4,Not applicable
6,role_evidence_sensitivity,5342,60.0
7,strict_provisional_sensitivity,60,60.0


Table C7. Analysis-ready views and call coverage.


## 11. Quality and unresolved cases

I retain **509 parser-warning records**: 53 ambiguous speaker labels, 41 layout-route records, one possible Q/A marker and 414 unparsed-text records. The complete queue has 530 rows, including 21 source/selection records. The table separates parser warnings from source/selection records. [Table C8]

The 61 source-revision quality observations report structural integrity separately from semantic accuracy. Independent call-level semantic verification remains **0/60**; analyst questions and Q&A links are structural proxies. BAC source status is provisional, publication-vintage fields are unknown and the corpus split is unassigned. [Tables C9–C10]


In [10]:
E = RELEASE / "E_quality_and_review"
review_queue = pd.read_parquet(E / "review_queue.parquet")
quality = pd.read_parquet(E / "quality_observations.parquet")

# Separate parser warnings from source/selection review records.
warning_counts = (
    review_queue.groupby(["record_type", "review_type"], dropna=False)
    .size().rename("rows").reset_index().sort_values("rows", ascending=False)
)
show_corpus_table(warning_counts, "Table C8. Complete review-record inventory.")

# Join bank IDs explicitly and summarise all revisions, including history.
quality_bank = quality.merge(
    source_revisions[["source_revision_id", "bank_id"]],
    on="source_revision_id", how="left", validate="one_to_one"
)
assert len(quality_bank) == len(source_revisions) == 61
assert quality_bank.bank_id.notna().all()
quality_summary = quality_bank.groupby([
    "bank_id", "processed_integrity_status", "human_semantic_status",
    "source_fidelity_status", "team_review_status"
], dropna=False).size().rename("source revisions").reset_index()
show_corpus_table(quality_summary, "Table C9. Quality states across all 61 source revisions.")

# Expose the availability/split fields behind the stated limitations.
metadata_state = pd.Series({
    "Source revisions": len(source_revisions),
    "Missing available_at": int(source_revisions.available_at.isna().sum()),
    "Corpus split values": ", ".join(sorted(source_revisions.corpus_split.astype(str).unique())),
    "Review queue records": len(review_queue),
    "Semantic status values": ", ".join(sorted(quality.human_semantic_status.astype(str).unique()))
})
show_corpus_table(metadata_state.to_frame("value"), "Table C10. Availability, split and semantic-status inventory.")


,record_type,review_type,rows
3,unparsed_text,CANONICAL_PARSER_REVIEW,414
0,ambiguous_speaker_label,CANONICAL_PARSER_REVIEW,53
1,layout_route,CANONICAL_PARSER_REVIEW,41
4,Not applicable,SOURCE_AND_SELECTION_STATE,21
2,possible_qa_layout_marker,CANONICAL_PARSER_REVIEW,1


Table C8. Complete review-record inventory.


,bank_id,processed_integrity_status,human_semantic_status,source_fidelity_status,team_review_status,source revisions
0,BAC,PASS,NOT_MEASURED,NOT_EVALUATED_FOR_PERSONAL_REVISION,PENDING_TEAM_DECISION,21
1,CITI,PASS,NOT_MEASURED,ACCEPTED_OFFICIAL_SOURCE,EXISTING_ACCEPTED_SOURCE_POLICY,20
2,JPM,PASS,NOT_MEASURED,ACCEPTED_OFFICIAL_SOURCE,EXISTING_ACCEPTED_SOURCE_POLICY,20


Table C9. Quality states across all 61 source revisions.


,value
Source revisions,61
Missing available_at,61
Corpus split values,UNASSIGNED
Review queue records,530
Semantic status values,NOT_MEASURED


Table C10. Availability, split and semantic-status inventory.


## 12. Downstream contract

I pass `question_units` to analyst-question modelling and `prepared_management_turns` to the prepared-remarks comparison. The 1,900-question population remains in attention accounting even when model fitting uses a subset.

I retain `call_id`, `turn_id`, `question_unit_id` and `exchange_id` when joining sentiment or other model outputs. The source/extraction links remain available for auditing.


## 13. Production entry points

From the project directory, these are the corresponding script interfaces. Paths in angle brackets are user-supplied inputs.

```bash
# Extraction and structural parsing
python scripts/extract_text.py --manifest <manifest.jsonl> --profile <profile>
python scripts/parse_transcripts.py --profile <profile>

# Offline reparse: requires the predecessor package and project dependencies
python scripts/controlled_60_reparse.py --workspace-root <workspace> --evidence-output <evidence.json>

# Historical candidate build: also requires its immutable Git authority
python scripts/r1c5_candidate_release.py build --workspace-root <workspace> --package-root <new-package-dir> --evidence-output <evidence.json> --build-source-commit 279b180cf6096857c9e379554785adb3470c11e4

# Candidate inventory: requires the private package-validation dependencies
python scripts/team_data_release.py --package-root team_data/CAM_BOE_TEAM_DATA_RELEASE_R1C5_CANDIDATE_V1 catalogue-candidate
```

These commands specify the production entry points. The executed cells in this notebook inspect the accepted package; Section 7 records the requirements for an exact rebuild.


## 14. Summary and limitations

My corpus contains **60 calls, 5,342 turns, 1,900 question units and 39,738 sentence rows**, with full three-bank coverage across 20 quarters. The 34 linked tables contain 77,070 logical rows, including overlapping research views.

The checks verify source identity, canonical counts and selected ID relationships. I retain unresolved records and the provisional BAC status. Exact historical reconstruction remains unverified until the missing build dependency is restored.

This notebook and the separate topic-attention notebook are my contributions; their sections can be combined using the team's preferred structure.
